In [1]:
!pip install z3-solver pulp


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np

In [3]:
# Generate synthetic data
N = 6
np.random.seed(37)
X = np.random.randint(1, 7, size=(N, 5))
X_petals = np.where(X == 3, 2, 0) + np.where(X == 5, 4, 0)
Y = X_petals.sum(axis=1)
X.shape, Y.shape

((6, 5), (6,))

In [4]:
# pivot to counts
C = np.array([np.bincount(row, minlength=7)[1:] for row in X])

In [ ]:
from scipy.optimize import leastsq

# Solve Cb = Y using numpy.linalg.solve (if C is square and invertible)
b = np.linalg.solve(C, Y)

solution = np.round(b).astype(int).tolist()
print("Solution:", solution)

# Note: this will fail if N !=6 or if any rows are linear combinations of others

Solution: [0, 0, 2, 0, 4, 0]


In [6]:
# Solve Cb = Y using least squares (leastsq)


def residuals(b, C, Y):
    return np.dot(C, b) - Y


b0 = np.zeros(C.shape[1])
b_hat, _ = leastsq(residuals, b0, args=(C, Y))
solution = np.round(b_hat).astype(int).tolist()
print("Solution:", solution)

Solution: [0, 0, 2, 0, 4, 0]


In [7]:
# sat solver
from z3 import IntVector, Solver, Sum, sat

# Cb = Y, where b is an integer vector of length 6
b = IntVector("b", 6)
solver = Solver()

# Add constraints: for each row in C, the dot product with b equals the corresponding Y
for i in range(N):
    solver.add(Sum([int(C[i, j]) * b[j] for j in range(6)]) == int(Y[i]))

if solver.check() == sat:
    model = solver.model()
    solution = [model[b_i].as_long() for b_i in b]
    print("Solution:", solution)

Solution: [0, 0, 2, 0, 4, 0]


In [8]:
from pulp import (
    PULP_CBC_CMD,
    LpInteger,
    LpMinimize,
    LpProblem,
    LpStatus,
    LpVariable,
    lpSum,
    value,
)

# Define the problem
prob = LpProblem("Find_b_coefficients", LpMinimize)

# Define integer variables b0...b5, all >= 0
b_vars = [LpVariable(f"b_{j}", lowBound=0, cat=LpInteger) for j in range(6)]

# Add constraints: for each row in C, the dot product with b_vars equals the corresponding Y
for i in range(N):
    prob += lpSum([int(C[i, j]) * b_vars[j] for j in range(6)]) == int(Y[i])

# Dummy objective: minimize the sum of coefficients
prob += lpSum(b_vars)

# Solve with CBC
prob.solve(PULP_CBC_CMD(msg=0))

# Extract solution
solution = [int(value(var)) for var in b_vars]
print("Problem Status:", LpStatus[prob.status])
print("Solution:", solution)

Problem Status: Optimal
Solution: [0, 0, 2, 0, 4, 0]
